In [ ]:
import numpy as np
import pandas as pd
from glob import glob

In [ ]:
with open('../scripts/preprocessing-stimulus/splits.txt') as f:
    splits = f.readlines()

In [ ]:
def convert_to_seconds(hhmmss):
    h, m, s = hhmmss.split(':')
    s = float(h) * 3600 + float(m) * 60 + float(s)
    return s

In [ ]:
splits = [s.strip().split() for s in splits]

In [ ]:
splits

In [ ]:
convert_to_seconds(splits[0][0])

In [ ]:
# I need to compute the following
# 1. Duration of the clip
# 2. Overlap of the old clip with new clip

In [ ]:
splits_s = np.array([[convert_to_seconds(ss) for ss in split] for split in splits])

In [ ]:
durations = np.round(np.diff(splits_s, 1))

In [ ]:
durations

In [ ]:
splits_s

In [ ]:
overlap = np.round(splits_s[:-1, 1] - splits_s[1:, 0])

In [ ]:
overlap = [0] + overlap.tolist()
overlap

In [ ]:
clip_start = 10.  # 10 s buffer at the beginning
dfs = []
for duration, over in zip(durations.ravel(), overlap):
    records = []
    onset = 0
    if over != 0:
        records.append({
            'onset': onset + clip_start,
            'duration': over,
            'trial_type': 'overlap_with_previous_run'
        })
        onset = onset + over
        duration = duration - over
    records.append({
        'onset': onset + clip_start,
        'duration': duration,
        'trial_type': 'movie'
    })
    dfs.append(pd.DataFrame.from_records(records))

In [ ]:
durations

In [ ]:
for df in dfs:
    print(df)

In [ ]:
n_trs_run = [
    598,
    498,
    535,
    618,
    803
]

In [ ]:
for df in dfs:
    print(df.onset[0] + np.sum(df.duration) + 10)

In [ ]:
subjects = sorted([s.split('/')[-1] for s in glob('../data/sub-*')])

In [ ]:
for s in subjects:
    print(s)
    events = sorted(glob(f"../data/{s}/func/*events.tsv"))
    for ev, df in zip(events, dfs):
        df.to_csv(ev, sep='\t', index=None)